In [2]:
import pandas as pd
from pathlib import Path

In [3]:
# set up
def find_root(marker: str = "pyproject.toml") -> Path:
    path = Path.cwd().resolve()
    while not (path / marker).exists():
        if path.parent == path:
            raise FileNotFoundError(f"No {marker} found above {Path.cwd()}")
        path = path.parent
    return path

ROOT = find_root()
RAW = ROOT / "data" / "raw" / "vegetation" / "ODK Data Exports"
ENT = ROOT / "data" / "raw" / "vegetation" / "Entity lists"
OUT = ROOT / "data" / "raw" / "output" / "vegetation"


# Read data

In [5]:
EXPECTED = {
    "herbaceous_veg_survey.csv": ["KEY", "ReviewState", "plot_selection-selected_plot_uuid",
                                  "survey_begin-start_time", "survey_end-end_time"],
    "herbaceous_veg_survey-quadrat_repeat.csv": ["KEY", "PARENT_KEY", "quadrat_number", "herbs_present",
                                                 "location_quadrat-Latitude", "location_quadrat-Longitude"],
    "herbaceous_veg_survey-additional_species_repeat.csv": ["KEY", "PARENT_KEY", "species_entry_mode",
                                                            "target_extra_name", "select_reuse_unknown",
                                                            "select_reuse_missing"],
    "register_vegetation_plots.csv": ["KEY", "plot_selection-selected_plot_uuid",
                                      "plot_selection-is_plot_viable"],
}

# first quality check: are all columns in the CSV as expected
def read(folder, name, expected=()):
    df = pd.read_csv(folder / name)
    missing = [c for c in expected if c not in df.columns]
    if missing:
        raise ValueError(f"{name}: missing columns {missing}")
    return df


survey_raw = read(RAW, "herbaceous_veg_survey.csv", EXPECTED["herbaceous_veg_survey.csv"])
quadrat_raw = read(RAW, "herbaceous_veg_survey-quadrat_repeat.csv",
                   EXPECTED["herbaceous_veg_survey-quadrat_repeat.csv"])
addsp_raw = read(RAW, "herbaceous_veg_survey-additional_species_repeat.csv",
                 EXPECTED["herbaceous_veg_survey-additional_species_repeat.csv"])
reg_raw = read(RAW, "register_vegetation_plots.csv", EXPECTED["register_vegetation_plots.csv"])

# these tables won't be updated frequently, so we're not checking the columns
species = read(ENT, "species.csv")
species_extra = read(ENT, "species_extra.csv")
species_ids = read(ENT, "species_extra_ids.csv")
vegplots = read(ENT, "vegplots.csv")
centroids = read(ENT, "centroids.csv")

# print shapes of all exported tables
for name, df in [("surveys", survey_raw), ("quadrats", quadrat_raw), ("additional species", addsp_raw),
                 ("plot registrations", reg_raw)]:
    print(f"{name:20s} {df.shape}")

surveys              (32, 52)
quadrats             (640, 21)
additional species   (1296, 30)
plot registrations   (31, 82)
